## Before you run a single cell in this notebook — terminal pre-flight

Open a JupyterLab terminal (**File > New > Terminal**) and run the commands below in order. Each guards against a failure mode that has actually cost money on prior runs. Do this BEFORE clicking any cell in the notebook.

### 1. Confirm the pod has enough disk

```bash
df -h /workspace
```

Look at the **Avail** column on the `/workspace` row. You need at least **40 GB free**. If you have less, terminate the pod and re-deploy with a larger Container Disk / Volume Disk size. Trying to train with less will hit "No space left on device" mid-run and you'll have to start over.

### 2. Confirm the GPU is the one you asked for, and nothing else is using it

```bash
nvidia-smi
```

The **Product Name** row should say `NVIDIA RTX 6000 Ada Generation`, `NVIDIA A40`, or `NVIDIA RTX A6000` (any 48 GB chip). The **Memory-Usage** column should show ~0 MB used and ~48 GB total. If a process is listed under the "Processes" section using GPU memory and you haven't started anything yet, it's a leftover from a previous session — kill it before launching:

```bash
nvidia-smi --query-compute-apps=pid --format=csv,noheader
kill -9 <pid_from_above>
```

### 3. Pull the latest code

```bash
cd /workspace/finpost
git status   # if anything shows as 'deleted:', run: git restore .
git pull --ff-only
```

### 4. Install the project (first time on this pod only)

```bash
pip install -e ".[dev]"
```

This takes 8–15 minutes. Long silences during the "Installing collected packages" phase are normal — do not interrupt it.

### 5. Confirm the install worked

```bash
python -c "import finpost; print(finpost.__file__)"
```

Expected output: `/workspace/finpost/src/finpost/__init__.py`. If you instead see `ModuleNotFoundError: No module named 'finpost'` despite the pip install succeeding, the editable install's PEP 660 hook didn't get written. Manual fix:

```bash
echo "/workspace/finpost/src" > /usr/local/lib/python3.11/dist-packages/finpost.pth
python -c "import finpost; print(finpost.__file__)"   # retry; should now print the path
```

### 6. Confirm torch sees the GPU

```bash
python -c "import torch; print('cuda:', torch.cuda.is_available())"
```

Expected output: `cuda: True`. If you see `cuda: False`, the installed torch wheel was built for a newer CUDA than the pod's driver supports. Downgrade torch to a CUDA 12.4 wheel:

```bash
pip install "torch==2.4.1+cu124" --index-url https://download.pytorch.org/whl/cu124 --force-reinstall --no-deps
python -c "import torch; print('cuda:', torch.cuda.is_available())"   # retry
```

### 7. Sanity-check one more time

```bash
df -h /workspace && nvidia-smi
```

Confirm disk and GPU still look healthy. You're now ready to run the notebook cells below in order.

---

Full troubleshooting table in [`docs/runbooks/runpod-end-to-end.md`](../docs/runbooks/runpod-end-to-end.md) (or the standalone `runpod-end-to-end.html` for offline reading). Subsequent kernel restarts on the same pod do not need re-installation.

---


# Phase 1 RunPod Ablation — Qwen 0.5B SFT @ 2000 steps

Three new SFT arms trained at 2000 steps each on the A40 RunPod, then evaluated on GSM8K + MATH via the exact-answer eval CLI. Designed to run in the pod's JupyterLab cell-by-cell.

**Why a new A40 recipe:** The 1000-step base-vs-combined run (`results/evals/base_vs_sft_run_2`) showed SFT had taught format (parse-success jumped 22% → 86% on GSM8K) but not capability (accuracy only +1.6 pp on GSM8K, **−15.8 pp on MATH**). That run also trained at `max_seq_len=512`, which truncates roughly 15% of MATH train docs (p95 of prompt+response is 784 tokens), so the training signal for MATH was structurally damaged — the model often never saw the `\boxed{...}` answer in those examples. This run tests whether a longer horizon plus an A40-appropriate recipe (longer packed sequences in particular) can learn actual math behavior rather than only answer shape.

**Three arms:**
- `gsm8k_only` — sources `[gsm8k]`, 2000 steps
- `math_only` — sources `[math]`, 2000 steps
- `combined` — sources `[gsm8k, math]`, 2000 steps

**Important comparison note:** this is a new experiment, not a strict continuation of the old Kaggle-compatible 1000-step recipe. The A40 recipe uses bf16, much larger micro-batches (bs=16 vs 1), longer packed sequences (1024 vs 512), a higher LR (2e-5 vs 1e-5), and more frequent checkpointing. Compare the three 2000-step arms to each other directly; reuse the saved base eval only as a reference line.

**Trajectory eval:** each arm saves checkpoints every 500 steps, and the eval cell converts and evaluates all four intermediate checkpoints per arm (step-500, 1000, 1500, 2000). That produces 12 (arm, step) rows in `accuracy_summary.csv` instead of 3, so you can see *when* MATH breaks, recovers, or plateaus — the headline phase-2 input.

**A40 recipe:** bf16, per-device batch **16**, grad accumulation 1, effective batch 16 (≈48 docs/step under greedy packing at seq=1024), lr 2e-5, max_seq_len 1024 (covers MATH p95 of 784 tokens), checkpoints every 500 steps (4 per arm). Eval uses batch size 128 for both GSM8K and MATH.

**Why bs=16:** the per-device batch goes up because Qwen 0.5B is small enough that an A40 sits mostly idle at bs=4 — bigger batches raise GPU utilization without dramatically increasing per-step wall time. Effective batch goes from 4 (≈12 docs/step) to 16 (≈48 docs/step), a sizeable but still SFT-typical effective batch.

**Why LR is held at 2e-5:** linear-scaling rule would push LR to ~8e-5 to match the 4× batch increase. We're holding it because stacking a 4× LR bump + larger batch + new dtype + new seq_len in one run makes any MATH movement uninterpretable. LR is the next single lever to pull if 2000 steps doesn't fix MATH.

**Time and cost estimate (A40 on-demand at $0.44/hr):**

| Step | Time | Cost |
|---|---|---|
| 3× SFT 2000 steps (bs=16, seq=1024) | ~45–90 min | ~$0.33–$0.66 |
| 12× checkpoint conversion | ~12 min | ~$0.09 |
| Eval (12 ckpts × 2 sources × 500) | ~30–45 min | ~$0.22–$0.33 |
| **Total** | **~1.5–2.5 h** | **~$0.65–$1.10** |

These are estimates. Trust the notebook's printed throughput and the eval CLI's cost ledger over this table.

**Before running this notebook:** make sure you've run `git pull --ff-only` in `/workspace/finpost` so the latest notebook, `convert_checkpoint_to_hf.py`, and eval CLI are present.

## Step 1 — Sanity-check the pod

GPU should be A40 with ~48 GB free and CUDA available. Disk should show **>=50 GB free** on `/workspace` — 4 checkpoints × 3 arms × ~3 GB each (model + optimizer state) plus HF-converted copies adds up to ~45 GB before eval. If either condition fails, terminate and re-deploy with a larger volume.

In [ ]:
!nvidia-smi
!df -h /workspace 2>/dev/null || df -h .   # confirm enough free space for checkpoints

In [ ]:
import os
import subprocess
import sys

# The notebook expects to live next to (or be uploaded into) /workspace/finpost.
# Switch into the repo so all later commands resolve from the package root.
REPO_ROOT = '/workspace/finpost'
os.chdir(REPO_ROOT)
print('cwd:', os.getcwd())

# Make sure we're on the latest main so the convert script + YAML conventions
# match what this notebook assumes.
subprocess.run(['git', '-C', REPO_ROOT, 'pull', '--ff-only'], check=True)
subprocess.run(['git', '-C', REPO_ROOT, 'log', '--oneline', '-1'], check=False)

# Activate the venv that the previous session set up. If this notebook is
# the first thing on a fresh pod, run pip install -e ".[dev]" in a terminal
# before continuing.
import torch
import finpost
print('\nfinpost:', finpost.__version__)
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## Step 2 — Ablation constants

All three arms share these hyperparameters. The only thing that differs is `sources` (which dataset(s) the arm trains on). This is the A40-optimized 2000-step recipe, so it should be compared arm-to-arm inside this notebook rather than treated as apples-to-apples with the old Kaggle fp32 run.


In [ ]:
# Training hyperparameters — A40-optimized 2000-step recipe.
ABLATION_STEPS = 2000
MAX_SEQ_LEN = 1024
DTYPE = 'bfloat16'
LR = 2.0e-5
PER_DEVICE_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 1
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
VAL_SPLIT_PCT = 5.0
SEED = 42

# Proportional warmup (10% of training). Validator requires warmup_steps < max_steps.
warmup_steps = max(5, ABLATION_STEPS // 10)

# Save mid-run so a late failure does not lose the whole arm, and so we can
# evaluate the full trajectory (not just the final checkpoint).
CHECKPOINT_EVERY_N_STEPS = 500
CHECKPOINT_RETENTION_LAST_N = 4

# Eval the trajectory, not just the endpoint. These are the step counts at
# which a checkpoint exists on disk (by virtue of CHECKPOINT_EVERY_N_STEPS).
# The eval cell converts and evaluates each one per arm.
INTERMEDIATE_STEPS = [500, 1000, 1500, 2000]

# Validation cadence: align with CHECKPOINT_EVERY_N_STEPS so val_loss is
# logged at the same steps the eval cell will later score.
VAL_EVERY_N_STEPS = CHECKPOINT_EVERY_N_STEPS

# Effective batch size = per_device_batch_size * grad_accum_steps.
effective_batch = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS

# Eval settings (applied after all three arms train).
# bs=128 fits comfortably on A40 at bf16 (~5 GB of 48 used). The eval CLI has
# automatic OOM halving (_generate_chunk_with_oom_fallback) so if a hard batch
# blows up we drop to 64 transparently. MATH responses have higher
# generation-time variance than GSM8K so straggler inefficiency caps the
# practical speedup, but headroom is there.
EVAL_N = 500
EVAL_SEED = 42
BATCH_SIZE_GSM8K = 128
BATCH_SIZE_MATH = 128
GPU_COST_PER_HOUR = 0.44

# Where artifacts go.
EXPERIMENTS_DIR = 'experiments/runpod_a40'
CHECKPOINTS_DIR = 'results/checkpoints'
EVAL_OUT_DIR = 'results/evals/ablation_a40_run_1'
EVAL_RUN_NAME = EVAL_OUT_DIR.rstrip('/').split('/')[-1]

print(f'ABLATION_STEPS:     {ABLATION_STEPS}')
print(f'warmup_steps:       {warmup_steps}')
print(f'per_device_batch:   {PER_DEVICE_BATCH_SIZE}')
print(f'effective batch:    {effective_batch}')
print(f'lr:                 {LR}')
print(f'max_seq_len:        {MAX_SEQ_LEN}')
print(f'dtype:              {DTYPE}')
print(f'checkpoint_every:   {CHECKPOINT_EVERY_N_STEPS}')
print(f'val_every:          {VAL_EVERY_N_STEPS}')
print(f'intermediate steps: {INTERMEDIATE_STEPS}')
print(f'eval batches:       gsm8k={BATCH_SIZE_GSM8K}, math={BATCH_SIZE_MATH}')


## Step 3 — Generate the three YAML configs

Written to `experiments/runpod_a40/` so they're reproducible and visible alongside the run.

In [ ]:
import yaml
from pathlib import Path

ARMS = {
    'gsm8k_only': ['gsm8k'],
    'math_only':  ['math'],
    'combined':   ['gsm8k', 'math'],
}

def build_config(arm_name: str, sources: list[str]) -> dict:
    """Build the trainer YAML for one arm. Hyperparameters from notebook constants."""
    run_name = f'qwen-{arm_name}-{ABLATION_STEPS}s-a40'
    return {
        'model': {
            'base_model_id': 'Qwen/Qwen2.5-0.5B',
            'dtype': DTYPE,
            'use_safetensors': True,
        },
        'data': {
            'sources': sources,
            'val_split_pct': VAL_SPLIT_PCT,
            'seed': SEED,
        },
        'training': {
            'max_steps': ABLATION_STEPS,
            'warmup_steps': warmup_steps,
            'lr': LR,
            'weight_decay': WEIGHT_DECAY,
            'grad_accum_steps': GRAD_ACCUM_STEPS,
            'grad_clip': GRAD_CLIP,
            # Validate at every checkpoint boundary so val_loss aligns with eval points.
            'val_every_n_steps': VAL_EVERY_N_STEPS,
            'checkpoint_every_n_steps': CHECKPOINT_EVERY_N_STEPS,
            'per_device_batch_size': PER_DEVICE_BATCH_SIZE,
        },
        'packing': {
            'max_seq_len': MAX_SEQ_LEN,
            'isolate_documents': True,
        },
        'logging': {
            'wandb_project': 'finpost-phase1-runpod',
            'run_name': run_name,
        },
        'checkpointing': {
            'save_dir': f'{CHECKPOINTS_DIR}/{run_name}',
            'retention_last_n': CHECKPOINT_RETENTION_LAST_N,
            'resume_from': None,
        },
    }

Path(EXPERIMENTS_DIR).mkdir(parents=True, exist_ok=True)

yaml_paths: dict[str, Path] = {}
for arm_name, sources in ARMS.items():
    cfg = build_config(arm_name, sources)
    yaml_path = Path(f'{EXPERIMENTS_DIR}/{arm_name}_{ABLATION_STEPS}_a40.yaml')
    yaml_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
    yaml_paths[arm_name] = yaml_path
    print(f'wrote {yaml_path}')

# Sanity-check by printing one of them so you can scan the values.
print('\n--- combined config ---')
print(yaml_paths['combined'].read_text())


In [ ]:
# Initialise the (arm, step) -> HF path mapping. Populated by the per-arm
# convert+cleanup cells below. Eval reads from this mapping at the end.
hf_paths: dict[tuple[str, int], str] = {}


## Step 3.5 — Canary: 50-step smoke test before paying for the full run

If this cell raises, do NOT continue to the training cells below — the recipe is broken on this hardware. Cost: ~$0.01. Saves you from a multi-arm production run that would die partway through.

In [ ]:
# Step 3.5 — 50-step canary on the combined arm
#
# This cell is the cheapest safety check we have. It writes a one-off YAML
# that mirrors the production recipe but runs for only 50 optimizer steps,
# then subprocess-launches the trainer against it. Pass criterion is
# subprocess exit code 0 — the trainer raises RuntimeError on non-finite
# loss (see `_check_finite_loss` in src/finpost/training/_guards.py), so a
# clean exit means no NaN appeared in 50 steps at production hyperparams.
#
# Cost: ~75 seconds of GPU time, ~$0.01. Worth it every run.
#
# IMPORTANT: the canary overrides BOTH `max_steps` AND `warmup_steps`.
# Inheriting the production `warmup_steps=200` would trip pydantic's
# `warmup_steps < max_steps` validator and produce a misleading config
# load error before the model ever starts training.

import subprocess

CANARY_STEPS = 50
canary_warmup = max(5, CANARY_STEPS // 10)

canary_cfg = build_config('combined', ARMS['combined'])
canary_cfg['training']['max_steps'] = CANARY_STEPS
canary_cfg['training']['warmup_steps'] = canary_warmup
canary_cfg['training']['val_every_n_steps'] = CANARY_STEPS  # one val pass at the end
canary_cfg['training']['checkpoint_every_n_steps'] = CANARY_STEPS  # one ckpt at end
canary_cfg['checkpointing']['save_dir'] = 'results/checkpoints/qwen-canary-50s-a40'
canary_cfg['checkpointing']['retention_last_n'] = 1
canary_cfg['logging']['run_name'] = 'qwen-canary-50s-a40'

canary_yaml = Path(f'{EXPERIMENTS_DIR}/canary_50_a40.yaml')
canary_yaml.write_text(yaml.safe_dump(canary_cfg, sort_keys=False), encoding='utf-8')
print(f'canary config: {canary_yaml}')
print(f'canary steps:  {CANARY_STEPS}   canary warmup: {canary_warmup}')
print()

result = subprocess.run(
    ['python', '-m', 'finpost.training.train', '--config', str(canary_yaml), '--device', 'cuda'],
    capture_output=True,
    text=True,
    check=False,
)
print('--- canary stdout (tail) ---')
print('\n'.join(result.stdout.splitlines()[-30:]))
if result.returncode != 0:
    print('--- canary stderr (tail) ---')
    print('\n'.join(result.stderr.splitlines()[-30:]))
    print()
    print('\n✗ CANARY FAILED — DO NOT launch full 2000-step run')
    print('   inspect stderr above; common causes:')
    print('     - Non-finite loss: trainer logic regression or hardware/SDPA issue')
    print('     - OOM: lower per_device_batch_size in hyperparams cell, retry canary')
    print('     - Config error: verify warmup_steps < max_steps in canary YAML')
    raise RuntimeError(f'Canary subprocess returned {result.returncode}; aborting.')

print()
print('\n✓ CANARY PASSED — safe to launch full 2000-step run\n')


In [ ]:
# Disk + GPU snapshot after the canary, before launching the full run.
# Healthy values: /workspace has ≥30 GB still free; nvidia-smi shows ~0 MB
# used (the canary subprocess freed its allocations on exit). If GPU shows
# residual memory, run `nvidia-smi --query-compute-apps=pid --format=csv,noheader`
# in a terminal and `kill -9 <pid>` before launching arm 1.
!df -h /workspace; nvidia-smi


## Step 4 — Train each arm sequentially

Three separate cells. Run them in order. The A40 recipe should be materially faster than the old Kaggle-compatible fp32/batch=1 path, but watch the actual tokens/sec in the output rather than trusting any estimate.

**If you need to stop mid-run:** Ctrl-C the cell. The trainer writes checkpoints every 500 steps and keeps all four trajectory checkpoints, so a late failure can resume from the most recent saved checkpoint rather than losing the entire arm. Set `resume_from: results/checkpoints/<run_name>/step-<NNNNNNNN>` in the YAML to resume.

**WANDB_MODE=offline** is set once for the whole notebook process — no cloud-auth dance, wandb writes to `wandb/offline-run-*/` for later sync if you want.

In [ ]:
# WANDB mode auto-detect.
#
# If WANDB_API_KEY is in the environment, switch wandb to ONLINE mode
# (writes runs to wandb.ai under your account). Otherwise, stay OFFLINE
# (writes to wandb/offline-run-*/ which you can sync later if you want).
#
# Set WANDB_API_KEY one of three ways before running this cell:
#   1. RunPod pod env var (set when creating the pod; persists for pod lifetime)
#   2. A .env file (gitignored; manually source before launching Jupyter)
#   3. Manual `export WANDB_API_KEY=...` in the JupyterLab terminal
#
# The wandb library itself looks up WANDB_API_KEY automatically; no extra
# auth call needed inside this notebook.
if os.environ.get('WANDB_API_KEY'):
    os.environ['WANDB_MODE'] = 'online'
    masked = os.environ['WANDB_API_KEY'][:4] + '...' + os.environ['WANDB_API_KEY'][-4:]
    print(f'WANDB_MODE = online   (using WANDB_API_KEY = {masked})')
else:
    os.environ['WANDB_MODE'] = 'offline'
    print('WANDB_MODE = offline  (no WANDB_API_KEY in env; runs land in wandb/offline-run-*/)')


### Arm 1 — gsm8k_only


In [ ]:
!python -m finpost.training.train \
    --config {yaml_paths['gsm8k_only']} \
    --device cuda

In [ ]:
# Convert gsm8k_only's intermediate checkpoints to HF format, then delete the
# raw step-* directories to free disk before the next arm trains. Each raw
# checkpoint is ~3 GB (weights + optimizer state); the HF copy is ~1 GB
# (weights only). We drop optimizer state because we don't need to RESUME
# from these steps — we only need to EVAL them. Cuts peak disk from ~48 GB
# to ~24 GB.
import shutil

arm_name = 'gsm8k_only'
run_name = f'qwen-{arm_name}-{ABLATION_STEPS}s-a40'

for step in INTERMEDIATE_STEPS:
    src = f'{CHECKPOINTS_DIR}/{run_name}/step-{step:08d}'
    dst = f'{CHECKPOINTS_DIR}/{run_name}-step{step}-hf'
    print(f'\n[{arm_name}] converting step-{step}:')
    print(f'  from: {src}')
    print(f'  to:   {dst}')
    !python scripts/convert_checkpoint_to_hf.py \
        --checkpoint-dir {src} \
        --out-dir {dst} \
        --dtype {DTYPE}
    hf_paths[(arm_name, step)] = dst

# All 4 conversions for this arm done — safe to drop the raw checkpoints.
for step in INTERMEDIATE_STEPS:
    raw = Path(f'{CHECKPOINTS_DIR}/{run_name}/step-{step:08d}')
    if raw.exists():
        shutil.rmtree(raw)
        print(f'[{arm_name}] deleted raw {raw}')

print(f'\n[{arm_name}] disk after cleanup:')
!df -h /workspace


### Arm 2 — math_only


In [ ]:
!python -m finpost.training.train \
    --config {yaml_paths['math_only']} \
    --device cuda

In [ ]:
# Convert math_only's intermediate checkpoints to HF format, then delete the
# raw step-* directories to free disk before the next arm trains. Each raw
# checkpoint is ~3 GB (weights + optimizer state); the HF copy is ~1 GB
# (weights only). We drop optimizer state because we don't need to RESUME
# from these steps — we only need to EVAL them. Cuts peak disk from ~48 GB
# to ~24 GB.
import shutil

arm_name = 'math_only'
run_name = f'qwen-{arm_name}-{ABLATION_STEPS}s-a40'

for step in INTERMEDIATE_STEPS:
    src = f'{CHECKPOINTS_DIR}/{run_name}/step-{step:08d}'
    dst = f'{CHECKPOINTS_DIR}/{run_name}-step{step}-hf'
    print(f'\n[{arm_name}] converting step-{step}:')
    print(f'  from: {src}')
    print(f'  to:   {dst}')
    !python scripts/convert_checkpoint_to_hf.py \
        --checkpoint-dir {src} \
        --out-dir {dst} \
        --dtype {DTYPE}
    hf_paths[(arm_name, step)] = dst

# All 4 conversions for this arm done — safe to drop the raw checkpoints.
for step in INTERMEDIATE_STEPS:
    raw = Path(f'{CHECKPOINTS_DIR}/{run_name}/step-{step:08d}')
    if raw.exists():
        shutil.rmtree(raw)
        print(f'[{arm_name}] deleted raw {raw}')

print(f'\n[{arm_name}] disk after cleanup:')
!df -h /workspace


### Arm 3 — combined


In [ ]:
!python -m finpost.training.train \
    --config {yaml_paths['combined']} \
    --device cuda

In [ ]:
# Convert combined's intermediate checkpoints to HF format, then delete the
# raw step-* directories to free disk before the next arm trains. Each raw
# checkpoint is ~3 GB (weights + optimizer state); the HF copy is ~1 GB
# (weights only). We drop optimizer state because we don't need to RESUME
# from these steps — we only need to EVAL them. Cuts peak disk from ~48 GB
# to ~24 GB.
import shutil

arm_name = 'combined'
run_name = f'qwen-{arm_name}-{ABLATION_STEPS}s-a40'

for step in INTERMEDIATE_STEPS:
    src = f'{CHECKPOINTS_DIR}/{run_name}/step-{step:08d}'
    dst = f'{CHECKPOINTS_DIR}/{run_name}-step{step}-hf'
    print(f'\n[{arm_name}] converting step-{step}:')
    print(f'  from: {src}')
    print(f'  to:   {dst}')
    !python scripts/convert_checkpoint_to_hf.py \
        --checkpoint-dir {src} \
        --out-dir {dst} \
        --dtype {DTYPE}
    hf_paths[(arm_name, step)] = dst

# All 4 conversions for this arm done — safe to drop the raw checkpoints.
for step in INTERMEDIATE_STEPS:
    raw = Path(f'{CHECKPOINTS_DIR}/{run_name}/step-{step:08d}')
    if raw.exists():
        shutil.rmtree(raw)
        print(f'[{arm_name}] deleted raw {raw}')

print(f'\n[{arm_name}] disk after cleanup:')
!df -h /workspace


In [ ]:
# Disk + GPU snapshot after all three arms have trained, converted, and
# cleaned up. Healthy values: /workspace has ≥30 GB still free; nvidia-smi
# shows ~0 MB used. If GPU shows residual memory, kill the offending
# process before launching the eval cells below.
!df -h /workspace; nvidia-smi


## Step 5 — Verify the converted HF checkpoints landed

The per-arm convert+cleanup cells above have already deleted the raw `step-*` directories, so verifying raw checkpoint existence no longer applies — we verify the HF-converted copies (which eval reads from) instead.

Every arm should have four HF-format directories on disk: `<run_name>-step{500,1000,1500,2000}-hf`. Any missing here is a hard failure rather than a warning.


In [ ]:
# Verify the HF-converted directory landed for every (arm, step). The
# per-arm convert+cleanup cells have already deleted the raw step-* dirs,
# so we verify the HF copies (the input to eval) instead of the raw
# checkpoints. Any missing dir here is a hard failure — eval can't run
# without it.
missing: list[str] = []
for arm_name in ARMS.keys():
    run_name = f'qwen-{arm_name}-{ABLATION_STEPS}s-a40'
    print(f'\n{arm_name}:')
    for step in INTERMEDIATE_STEPS:
        hf_dir = Path(f'{CHECKPOINTS_DIR}/{run_name}-step{step}-hf')
        ok = hf_dir.exists()
        marker = 'OK ' if ok else 'MISSING'
        print(f'  step-{step:>5} hf: {marker}  {hf_dir}')
        if not ok:
            missing.append(str(hf_dir))
        else:
            files = sorted(p.name for p in hf_dir.iterdir())
            print(f'    files: {files}')

if missing:
    raise FileNotFoundError(f'Missing HF checkpoint dirs: {missing}')


## Step 6 — Run eval across all 12 (arm, step) checkpoints

One CLI call across the twelve HF-format checkpoints on both sources. Same seed and n as the previous base eval, so per-example details are comparable without rerunning base.

This eval intentionally excludes `base=Qwen/Qwen2.5-0.5B`. Reuse `results/evals/base_vs_sft_run_2/accuracy_summary.csv` as the base reference line offline. Eval batch is 128 — comfortably fits on A40 memory-wise; if a particular chunk OOMs the CLI's `_generate_chunk_with_oom_fallback` halves transparently.

In [ ]:
# Eval CLI expects --checkpoints name=path entries. We build one entry per
# (arm, step) pair so the resulting accuracy_summary.csv has 12 rows per source
# (3 arms x 4 step counts), which is the trajectory we want for analysis.
checkpoint_args = ' '.join(
    f'{arm}-{step}={path}' for (arm, step), path in hf_paths.items()
)
print('--checkpoints', checkpoint_args)
print()

!python -m finpost.evals.eval_exact \
    --checkpoints {checkpoint_args} \
    --sources gsm8k math \
    --n {EVAL_N} --seed {EVAL_SEED} \
    --out-dir {EVAL_OUT_DIR} \
    --batch-size-gsm8k {BATCH_SIZE_GSM8K} --batch-size-math {BATCH_SIZE_MATH} \
    --gpu-cost-per-hour {GPU_COST_PER_HOUR} \
    --device cuda


## Step 7 — Display the headline numbers

In [ ]:
import json
from pathlib import Path

summary_csv = Path(EVAL_OUT_DIR) / 'accuracy_summary.csv'
cost_json = Path(EVAL_OUT_DIR) / 'cost_summary.json'

print('=== accuracy_summary.csv ===')
print(summary_csv.read_text())

print('\n=== cost_summary.json ===')
print(cost_json.read_text())

In [ ]:
# Optional — also load the previous run's base numbers for an inline trajectory table.
try:
    import pandas as pd
    df_new = pd.read_csv(summary_csv)
    # Split checkpoint label `<arm>-<step>` into separate columns for readability.
    # rsplit on the LAST '-' so multi-segment arm names (e.g. `gsm8k_only`) survive.
    df_new[['arm', 'step']] = df_new['checkpoint'].str.rsplit('-', n=1, expand=True)
    df_new['step'] = df_new['step'].astype(int)

    print('\n=== Ablation trajectory @ 2000 steps (n=500 per cell) ===')
    print(
        df_new[['arm', 'step', 'source', 'accuracy', 'parse_success_rate']]
        .sort_values(['source', 'arm', 'step'])
        .to_string(index=False)
    )

    base_csv = Path('results/evals/base_vs_sft_run_2/accuracy_summary.csv')
    if base_csv.exists():
        df_base = pd.read_csv(base_csv)
        df_base = df_base[df_base['checkpoint'] == 'base'].copy()
        df_base['arm'] = 'base'
        df_base['step'] = 0
        df_all = pd.concat(
            [df_base[df_new.columns], df_new], ignore_index=True
        )
        print('\n=== With base (from base_vs_sft_run_2) ===')
        print(
            df_all[['arm', 'step', 'source', 'accuracy', 'parse_success_rate']]
            .sort_values(['source', 'arm', 'step'])
            .to_string(index=False)
        )
except ImportError:
    print('pandas not available — read the CSV manually above.')


## Step 8 — Tar the eval results for download

Bundles the eval output into a single tarball. Then in JupyterLab's file browser, navigate to `results/evals/`, right-click `ablation_a40_run_1.tar.gz`, and click **Download**.


In [ ]:
!tar -czf {EVAL_OUT_DIR}.tar.gz -C results/evals {EVAL_RUN_NAME}
!ls -lah {EVAL_OUT_DIR}.tar.gz


## Step 9 — (Optional) push the three final checkpoints to HF Hub

Useful if you want to re-eval them later from a different pod without retraining. Each is ~2 GB. Skip this if you don't plan to come back to these specific checkpoints — the volume disk on a stopped pod keeps them around for ~$0.14/day anyway.

In [ ]:
# Uncomment to push only the final 2000-step HF checkpoints.
# Requires `hf auth login` with a Write token first.
# for (arm_name, step), hf_dir in hf_paths.items():
#     if step != ABLATION_STEPS:
#         continue
#     repo_id = f'sl891/finpost-sft-{arm_name}-{ABLATION_STEPS}-a40'
#     print(f'\n=== Uploading {arm_name} step-{step} -> {repo_id} ===')
#     !hf upload {repo_id} {hf_dir}


## Final — Stop the pod

Download the tarball via the JupyterLab file browser, then in the RunPod console:

- **Stop Pod** — billing stops, volume persists. Use this if you'll be back within a few days.
- **Terminate Pod** — frees everything, including the venv and checkpoints. Use this only after all eval outputs and checkpoints you care about are downloaded or pushed.

Record the final cost from `cost_summary.json`; treat the estimates at the top of this notebook as planning numbers only.
